# 07 --- MCP Primitives

**CCA Pattern**: MCP (Model Context Protocol) servers expose three primitives:
- **Tools** -- executable functions an agent can invoke (verbs)
- **Resources** -- data the application can read and hand to the model (nouns)
- **Prompts** -- reusable, parameterized message templates (patterns)

The exam tests whether you know the difference.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
import json

from research_agents.tools.definitions import ALL_TOOL_SETS
from research_agents.tools.handlers import dispatch
from research_agents.agent.subagents import WEB_RESEARCHER_PROMPT
from research_agents.services.container import make_default_services

services = make_default_services()

## Understanding the Three MCP Primitives

MCP is a protocol that standardizes how AI agents interact with external systems.
The CCA exam tests your ability to classify capabilities into the correct primitive.

### The Key Distinction

| Primitive | Nature | Analogy | Example |
|-----------|--------|---------|--------|
| **Tool** | Action (verb) | Function call | `verify_claim(claim)` |
| **Resource** | Data (noun) | Database query | Source reliability ratings |
| **Prompt** | Template (pattern) | Reusable format | Research query decomposition |

### Who is in control

The MCP specification's strongest classification tell is *who decides to use
the primitive* (Server features, "Control hierarchy"):

| Primitive | Controlled by | Meaning |
|-----------|---------------|---------|
| **Prompts** | User | Exposed for the user to pick explicitly (slash commands, menus) |
| **Resources** | Application | The host app decides what context to attach for the model |
| **Tools** | Model | The model decides to call them, subject to human approval |

The most common exam mistake is classifying a **Resource** as a **Tool**.
A source reliability database is *data the application can read* (Resource),
not *an action the model chooses to perform* (Tool).

### How Our System Maps to MCP

Our project uses the `anthropic` SDK directly (not the Agent SDK), so MCP primitives
are represented as plain Python constructs. Here's how each maps:

## Tools (Verbs) -- Things the Agent Does

In [ ]:
# Our tool definitions are MCP Tools -- executable functions
for agent_type, tools in ALL_TOOL_SETS.items():
    print(f'{agent_type}:')
    for tool in tools:
        print(f'  Tool: {tool["name"]:<25s} (action/verb)')

Each tool above is an **action** the agent can invoke. `search_web` performs a search.
`verify_claim` checks a claim. `delegate_task` triggers delegation. These are verbs.

In MCP terms, each tool definition has:
- A `name` (the function to call)
- A `description` (what it does and does NOT do)
- An `input_schema` (the parameters it accepts)

Calling one performs computation and returns a result:

In [ ]:
verdict = json.loads(dispatch('fact_checker', 'verify_claim',
                              {'claim': 'Remote workers are 13% more productive'}, services))
print(json.dumps(verdict['data'], indent=2))

## Resources (Nouns) -- Things the Agent Reads

In our system these are read-only service methods. In an MCP server each would
be exposed as a resource identified by a URI. The scheme is the server's
choice -- `file://`, `https://`, or something custom like `kb://` -- and the
client reads it with `resources/read`. Reading a resource has no side effects.

In [ ]:
kb = services.knowledge_base
docs = services.document_store
db = services.database

resources = {
    'kb://sources/reliability': kb.get_source_reliability('https://energy.gov/renewable-2024').value,
    'docs://catalog': [d.doc_id for d in docs.list_documents()],
    'db://schemas/remote_work_stats': [c['name'] for c in db.get_schema('remote_work_stats')['columns']],
}
for uri, value in resources.items():
    print(f'  Resource {uri:<32s} -> {value}')

These are **data** the application queries, not actions the model performs.
In our codebase:

- `kb://sources/reliability` -> `KnowledgeBase.get_source_reliability()`
returns a `SourceReliability` enum
- `docs://catalog` -> `DocumentStore.list_documents()` returns document metadata
- `db://schemas/...` -> `DatabaseService.get_schema()` returns column definitions

## Prompts (Patterns) -- Templates the Agent Uses

In [ ]:
# Prompts in our system (conceptual mapping)
prompts = {
    'research_query_template': 'Template for decomposing a query into subtasks',
    'citation_format_template': 'Template for formatting citations (APA)',
    'error_report_template': 'Template for reporting structured errors',
}
for name, desc in prompts.items():
    print(f'  Prompt: {name:<30s} -- {desc}')
print()
print('Closest thing in this codebase (a system prompt, not an MCP Prompt):')
print(WEB_RESEARCHER_PROMPT[:160].strip() + '...')

In our codebase, these map to:

- `research_query_template` -> The system prompts in `subagents.py`
(e.g., `WEB_RESEARCHER_PROMPT`)
- `citation_format_template` -> Would be an MCP Prompt if we needed
configurable citation styles
- `error_report_template` -> The `ToolErrorResponse` model structure

In MCP, Prompts are reusable templates that can be invoked with parameters.
They're not system prompts -- they're parameterized message templates
that standardize common operations, and the *user* chooses when to apply one.

## Classification Exercise

Five items below. For each, decide whether it is a **Tool**, a **Resource**,
or a **Prompt** *before* reading the worked answer in the cell that follows.
The CCA exam rewards snap classification, so practice making the call first.

### Item 1 -- `verify_claim(claim: str)` returning `{verified, confidence, source}`

Your classification?

<details>
<summary>Reveal the answer</summary>

**Tool.** It performs an action: looking the claim up in the knowledge base,
ranking the matches, and scoring the result. The model decides when to call
it. Anything shaped like a function call with an imperative verb in the name
is almost always a Tool.

Trap to avoid: candidates sometimes classify it as a Resource because it
"returns information." Tools also return information -- the distinguishing
feature is *computation the model triggers*, not read-only access.

</details>

### Item 2 -- A database of source URLs with reliability ratings

Your classification?

<details>
<summary>Reveal the answer</summary>

**Resource.** It is read-only data. In MCP, the server would expose it under a
URI of its choosing (say `kb://sources/reliability`) and the application would
*read* it with `resources/read` rather than have the model *call* it.

This is **the** canonical exam trap. Options include
`get_source_reliability(url)` (a Tool-like wrapper) as a distractor. The
underlying capability is a Resource even if the access pattern looks
Tool-shaped. The exam question is usually worded in terms of the data
itself ("a database of ratings"), which is the tell.

</details>

### Item 3 -- A template for decomposing a research query into SubTasks

Your classification?

<details>
<summary>Reveal the answer</summary>

**Prompt.** It is a reusable pattern, parameterized by the incoming query,
that produces a structured decomposition. Prompts in MCP are not system
prompts -- they are *parameterized message templates* a user can invoke to
standardize common operations.

If you said "Tool" because it produces an output: Prompts also produce
outputs. The distinguishing feature of a Prompt is that it is a
*template* with named parameters, not a function with computation.

</details>

### Item 4 -- `fetch_page(url: str) -> PageContent`

Your classification?

<details>
<summary>Reveal the answer</summary>

**Tool.** It performs the action of fetching a page and has observable side
effects (network call, possible failure modes). This is the easiest kind of
item on the exam -- clear verb, clear parameter, clear return type. If it
can time out or fail, it is almost certainly a Tool (Resources are usually
read-only and cacheable in a way that fetch calls are not).

</details>

### Item 5 -- A catalog listing available research documents with metadata

Your classification?

<details>
<summary>Reveal the answer</summary>

**Resource.** A catalog is data the agent browses. The items in it may be
things the agent can act on, but the catalog *itself* is a Resource.

Watch this distinction on the exam: `list_documents()` is often listed as a
Tool, but the underlying "available documents" is the Resource. If the
question asks about the *capability*, go Resource. If it asks about the
*function exposed to the agent*, the answer might be Tool. Read carefully.

</details>

## CCA Exam Tip

> The MCP primitives question is a vocabulary test:
> - **Tools** are things the agent *does* (verbs) -- model-controlled
> - **Resources** are things the application *reads* for the model (nouns) -- application-controlled
> - **Prompts** are templates the *user* invokes (patterns) -- user-controlled
>
> If a question describes a source reliability database -> **Resource**, not a Tool.
> If it describes a function that verifies a claim -> **Tool**.
> If it describes a reusable template for formatting citations -> **Prompt**.
>
> The most common distractor makes a Resource look like a Tool because
both involve 'getting information.' The difference: Tools have **side effects**
or perform **computation**; Resources are **read-only data**.

*Foolproof test:* **if you cannot *run* it, it is probably a Resource.**
A Tool *performs an action* (e.g., `fetch_page(url)`, `verify_claim(claim)`)
-- often *using* a Resource. A Resource is the *data or infrastructure the
Tool acts upon* (e.g., a database of sources, a schema catalog, a file
system). Same information can appear behind either primitive -- ask which
side of the verb it sits on, and who decides to use it.